# 🏋️‍♂️ Week 3: Python & Data Wrangling

**Data Analyst Course — Practical Assignment**  
**Core Topic**: Data Cleaning, Wrangling, Feature Engineering & Exploratory Visualizations  
**Dataset**: Fitness / Workout Tracking Dataset (`dirty_data.csv`)  
**Libraries**: `pandas`, `numpy`, `matplotlib`, `seaborn`

---

### 🎯 Assignment Objectives:
1. **Load and audit a messy dataset** in Pandas.
2. **Handle missing values and date inconsistencies**.
3. **Detect and fix outliers / typos** in numerical attributes.
4. **Remove duplicate records**.
5. **Filter rows** based on analytical fitness criteria.
6. **Engineer new features** (`Calories_Per_Minute`, `Heart_Rate_Spread`, `Workout_Intensity`, `Day_Name`).
7. **Create informative data visualizations** using Matplotlib & Seaborn.

## 📦 Step 1: Import Libraries & Environment Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling setup
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)
print('Environment and libraries ready!')

## 🔍 Step 2: Load Messy Dataset & Initial Data Audit

In [2]:
raw_path = os.path.join('..', 'data', 'dirty_data.csv')
df = pd.read_csv(raw_path)

print(f'Raw Shape: {df.shape[0]} rows, {df.shape[1]} columns\n')
df.head(10)

In [3]:
# Check data types and non-null counts
df.info()

In [4]:
# Audit missing values
print('Missing Values per Column:')
print(df.isnull().sum())

# Audit duplicate rows
print(f'\nTotal Duplicate Rows: {df.duplicated().sum()}')

## 🧹 Step 3: Clean & Standardize Date Column
- Remove extraneous quote marks.
- Impute missing sequential calendar date for index 22 (`2020-12-22`).
- Convert strings and integer date formats (`20201226`) into standardized datetime objects.

In [5]:
# Strip quotes from string dates
df['Date'] = df['Date'].astype(str).str.replace("'", "").str.strip()

# Detect and impute missing date
missing_date_mask = df['Date'].isna() | df['Date'].isin(['', 'nan', 'None'])
df.loc[missing_date_mask, 'Date'] = '2020-12-22'

# Convert to datetime
df['Date'] = pd.to_datetime(df['Date'], format='mixed')
print('Date column converted successfully:')
df[['Date', 'Duration', 'Pulse', 'Calories']].head(5)

## ⚠️ Step 4: Detect & Fix Outliers in Duration
- Row 7 contains a `Duration` of `450` minutes with only 253.3 kcal burned.
- This is an obvious typographical error for `45` minutes.

In [6]:
# Inspect outlier
print('Outliers in Duration (> 120 min):')
print(df[df['Duration'] > 120])

# Correct outlier
df.loc[df['Duration'] == 450, 'Duration'] = 45
print('\nOutlier corrected. Duration summary:')
print(df['Duration'].describe())

## 🩹 Step 5: Handle Missing Values in Calories
- Impute missing values using the duration-specific median calories.

In [7]:
# Impute missing Calories with median grouped by Duration
duration_median = df.groupby('Duration')['Calories'].transform('median')
df['Calories'] = df['Calories'].fillna(duration_median).fillna(df['Calories'].median())
df['Calories'] = df['Calories'].round(1)

print(f'Remaining missing Calories: {df["Calories"].isnull().sum()}')

## 🗑️ Step 6: Remove Duplicate Records

In [8]:
initial_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {initial_rows - len(df)} duplicate row(s). Final row count: {len(df)}')

## ⚙️ Step 7: Feature Engineering (Creating New Columns)
- `Calories_Per_Minute`: Rate of energy expenditure.
- `Heart_Rate_Spread`: `Maxpulse - Pulse` (cardiovascular dynamic range).
- `Workout_Intensity`: Tiered category (`High`, `Moderate`, `Light`).
- `Day_Name`: Day of week extracted from `Date`.
- `Is_Weekend`: Weekend flag.

In [9]:
# 1. Calories per Minute
df['Calories_Per_Minute'] = (df['Calories'] / df['Duration']).round(2)

# 2. Heart rate spread
df['Heart_Rate_Spread'] = df['Maxpulse'] - df['Pulse']

# 3. Intensity categorization
conditions = [
    (df['Pulse'] >= 110) | (df['Calories_Per_Minute'] >= 6.0),
    (df['Pulse'] >= 100) & (df['Pulse'] < 110),
    (df['Pulse'] < 100)
]
choices = ['High Intensity', 'Moderate Intensity', 'Light Intensity']
df['Workout_Intensity'] = np.select(conditions, choices, default='Moderate Intensity')

# 4. Day of the week
df['Day_Name'] = df['Date'].dt.day_name()
df['Is_Weekend'] = df['Date'].dt.dayofweek.isin([5, 6])

df.head()

## 🔍 Step 8: Row Filtering Demonstrations
Filter subsets of records matching key workout performance thresholds.

In [10]:
# Filter 1: Workouts with over 300 kcal burned
high_cal = df[df['Calories'] > 300]
print(f'Workouts with > 300 kcal: {len(high_cal)} sessions')

# Filter 2: Extended sessions (>= 60 minutes)
long_sessions = df[df['Duration'] >= 60]
print(f'Workouts >= 60 min: {len(long_sessions)} sessions')

# Filter 3: Peak cardio heart rate (Maxpulse >= 140 bpm)
peak_cardio = df[df['Maxpulse'] >= 140]
print(f'Workouts with Maxpulse >= 140: {len(peak_cardio)} sessions')
peak_cardio[['Date', 'Duration', 'Pulse', 'Maxpulse', 'Calories', 'Workout_Intensity']]

## 📊 Step 9: Exploratory Data Visualization (Matplotlib & Seaborn)

In [11]:
# Visual 1: Scatter plot with regression line
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df, 
    x='Duration', 
    y='Calories', 
    color='#1F4E79', 
    scatter_kws={'s': 65, 'alpha': 0.8},
    line_kws={'color': '#D9534F', 'linewidth': 2}
)
plt.title('Workout Duration vs. Calories Burned', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Session Duration (Minutes)')
plt.ylabel('Total Calories Burned (kcal)')
plt.tight_layout()
plt.show()

In [12]:
# Visual 2: Workout Intensity Boxplot
plt.figure(figsize=(8, 5))
order = ['Light Intensity', 'Moderate Intensity', 'High Intensity']
sns.boxplot(data=df, x='Workout_Intensity', y='Calories', order=order, palette=['#A2D2FF', '#BDE0FE', '#FFC8DD'])
sns.stripplot(data=df, x='Workout_Intensity', y='Calories', order=order, color='#2B2D42', size=7, jitter=0.2, alpha=0.7)
plt.title('Calorie Burn by Workout Intensity Tier', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Workout Intensity Tier')
plt.ylabel('Calories Burned (kcal)')
plt.tight_layout()
plt.show()

In [13]:
# Visual 3: Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
num_cols = ['Duration', 'Pulse', 'Maxpulse', 'Calories', 'Calories_Per_Minute', 'Heart_Rate_Spread']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='Blues', square=True, linewidths=1)
plt.title('Health & Fitness Metrics Correlation Heatmap', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

## 💾 Step 10: Export Cleaned Dataset

In [14]:
out_path = os.path.join('..', 'data', 'cleaned_data.csv')
df.to_csv(out_path, index=False)
print(f'Cleaned data successfully saved to: {out_path}')